# Demo 단기 메모리 생성

> **목적**: GCS 실데이터(house_011) 기반으로 단기 메모리 JSON 생성  
> → `memory/short_term/house_011.json` 저장

**입력**: GCS `training_dev10/` + `labels/training.parquet`  
**출력**: `{ShortTermEvent 리스트}` JSON  
**의존**: `cold_start/reference_images.json` (TDA 모드 분류용, 없으면 W 범위 룩업 폴백)

In [ ]:
!pip install -q gcsfs pyarrow

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
print('GCP 인증 + Drive 마운트 완료')

In [ ]:
import json
import warnings
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import gcsfs
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')
print('임포트 완료')

In [ ]:
BUCKET_PREFIX = 'ax-nilm-data-dhwang0803-us/nilm/training_dev10'
LABEL_PATH    = 'ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet'

DEMO_HOUSE    = 'house_011'   # 데모 대상 가구

COLD_START_DIR = Path('/content/drive/MyDrive/ax_nilm_cold_start')
REF_PATH       = COLD_START_DIR / 'reference_images.json'
OUTPUT_PATH    = COLD_START_DIR / f'short_term_{DEMO_HOUSE}.json'

# W 범위 모드 분류용 thresholds (build_tda_references.ipynb와 동일)
THRESHOLDS = {
    '에어컨': [
        {'name': 'fan_low',     'min_w': 0.0,    'max_w': 11.6},
        {'name': 'cool_medium', 'min_w': 11.6,   'max_w': 20.6},
        {'name': 'cool_high',   'min_w': 20.6,   'max_w': None},
    ],
    '김치냉장고': [
        {'name': 'fan',       'min_w': 0.0,    'max_w': 68.7},
        {'name': 'cool_low',  'min_w': 68.7,   'max_w': 129.1},
        {'name': 'cool_high', 'min_w': 129.1,  'max_w': None},
    ],
    '제습기': [
        {'name': 'fan_only',     'min_w': 0.0,    'max_w': 96.6},
        {'name': 'dehumid_low',  'min_w': 96.6,   'max_w': 229.6},
        {'name': 'dehumid_high', 'min_w': 229.6,  'max_w': None},
    ],
    '세탁기': [
        {'name': 'wash', 'min_w': 0.0,    'max_w': 169.2},
        {'name': 'spin', 'min_w': 169.2,  'max_w': None},
    ],
    '의류건조기': [
        {'name': 'standby',  'min_w': 0.0,    'max_w': 253.0},
        {'name': 'drum',     'min_w': 253.0,  'max_w': 620.0},
        {'name': 'dry_mid',  'min_w': 620.0,  'max_w': 1271.0},
        {'name': 'dry_high', 'min_w': 1271.0, 'max_w': None},
    ],
    '일반냉장고': [
        {'name': 'standby', 'min_w': 0.0,   'max_w': 52.0},
        {'name': 'cool',    'min_w': 52.0,  'max_w': 172.0},
        {'name': 'defrost', 'min_w': 172.0, 'max_w': None},
    ],
    '식기세척기': [
        {'name': 'rinse',    'min_w': 0.0,     'max_w': 419.4},
        {'name': 'wash',     'min_w': 419.4,   'max_w': 1171.2},
        {'name': 'heat_dry', 'min_w': 1171.2,  'max_w': None},
    ],
    '온수매트': [
        {'name': 'low',    'min_w': 0.0,    'max_w': 130.8},
        {'name': 'medium', 'min_w': 130.8,  'max_w': 301.6},
        {'name': 'high',   'min_w': 301.6,  'max_w': None},
    ],
    '전기밥솥': [
        {'name': 'keep_warm', 'min_w': 0.0,    'max_w': 608.8},
        {'name': 'cook',      'min_w': 608.8,  'max_w': None},
    ],
    '전기장판': [
        {'name': 'low',  'min_w': 0.0,  'max_w': 76.5},
        {'name': 'high', 'min_w': 76.5, 'max_w': None},
    ],
}

# 코드 가전명 → thresholds 키 매핑
_KEY_MAP = {
    '일반 냉장고':       '일반냉장고',
    '김치 냉장고':       '김치냉장고',
    '식기세척기/건조기': '식기세척기',
    '전기장판, 담요':    '전기장판',
    '진공 청소기(유선)': '진공청소기',
    '인덕션(전기레인지)': '인덕션',
}

ON_THRESHOLD = 10.0   # W 이하는 OFF로 판단
STANDBY_MIN_W   = 1.0
STANDBY_MIN_MIN = 30.0

print(f'데모 가구: {DEMO_HOUSE}')
print(f'출력 경로: {OUTPUT_PATH}')

In [ ]:
gcs   = gcsfs.GCSFileSystem()
pa_fs = pa.fs.PyFileSystem(pa.fs.FSSpecHandler(gcs))

labels_df = pq.read_table(LABEL_PATH, filesystem=pa_fs).to_pandas()

# 데모 가구 채널 목록
house_channels = labels_df[
    labels_df['household_id'] == DEMO_HOUSE
][['appliance_name', 'channel']].drop_duplicates()

print(f'{DEMO_HOUSE} 가전 목록:')
for _, row in house_channels.iterrows():
    print(f'  {row.appliance_name} / {row.channel}')

In [ ]:
import io
from concurrent.futures import ThreadPoolExecutor


def load_raw_channel(house_id, channel):
    periods = get_on_periods(house_id, channel)
    if not periods:
        return pd.DataFrame(columns=['date_time', 'active_power'])

    on_dates = set()
    for s, e in periods:
        d = s.date()
        while d <= e.date():
            on_dates.add(d.strftime('%Y%m%d'))
            d += pd.Timedelta(days=1)

    path = f'{BUCKET_PREFIX}/household_id={house_id}/channel={channel}'
    files = []
    for date in sorted(on_dates):
        try:
            files.extend(gcs.ls(f'{path}/date={date}'))
        except Exception:
            pass

    if not files:
        return pd.DataFrame(columns=['date_time', 'active_power'])

    def _read(f):
        with gcs.open(f, 'rb') as fp:
            buf = fp.read()
        return pq.read_table(io.BytesIO(buf), columns=['date_time', 'active_power']).to_pandas()

    with ThreadPoolExecutor(max_workers=8) as pool:
        dfs = list(pool.map(_read, files))

    df = pd.concat(dfs, ignore_index=True)
    df['date_time'] = pd.to_datetime(df['date_time']).dt.tz_localize(None)
    return df.sort_values('date_time').reset_index(drop=True)


def get_on_periods(house_id, channel):
    rows = labels_df[
        (labels_df['household_id'] == house_id) &
        (labels_df['channel'] == channel)
    ]
    periods = []
    for _, row in rows.iterrows():
        s = pd.to_datetime(row.get('start_ts'))
        e = pd.to_datetime(row.get('end_ts'))
        if pd.notna(s) and pd.notna(e):
            s = s.tz_convert('UTC').tz_localize(None) if s.tzinfo else s
            e = e.tz_convert('UTC').tz_localize(None) if e.tzinfo else e
            periods.append((s, e))
    return periods


def classify_mode_by_w(appliance_name, power_w):
    key    = _KEY_MAP.get(appliance_name, appliance_name)
    states = THRESHOLDS.get(key, [])
    for state in states:
        lo = state['min_w']
        hi = state.get('max_w')
        if lo <= power_w and (hi is None or power_w < hi):
            return state['name']
    return 'unknown'


def make_short_term_event(appliance, seg, timestamps, standby=None):
    sample_s   = pd.Series(timestamps).diff().median().total_seconds()
    sample_h   = sample_s / 3600
    avg_w      = float(seg.mean())
    mode       = classify_mode_by_w(appliance, avg_w)

    event = {
        'appliance':       appliance,
        'mode':            mode,
        'started_at':      timestamps[0].isoformat(),
        'duration_min':    round(len(seg) * sample_h * 60, 1),
        'energy_wh':       round(float(seg.sum() * sample_h), 3),
        'avg_w':           round(avg_w, 2),
        'peak_w':          round(float(seg.max()), 2),
        'tda_fingerprint': None,
        'standby':         standby,
    }
    return event


print('헬퍼 함수 정의 완료')

In [ ]:
short_term_events = []

for _, row in house_channels.iterrows():
    app_name = row['appliance_name']
    channel  = row['channel']

    print(f'\n[{app_name} / {channel}] 로드 중...')
    raw     = load_raw_channel(DEMO_HOUSE, channel)
    periods = get_on_periods(DEMO_HOUSE, channel)
    if not periods or raw.empty:
        print(f'  ON 구간 없음, 스킵')
        continue

    sample_s = raw['date_time'].diff().median().total_seconds()
    raw_idx  = raw.set_index('date_time').sort_index()

    for s, e in periods:
        seg = raw_idx.loc[s:e, 'active_power'].values.astype(np.float32)
        ts  = raw_idx.loc[s:e].index.tolist()
        if len(seg) < 2:
            continue

        # 대기전력 감지 — ON 직전 2시간 윈도우
        standby = None
        pre_start = s - pd.Timedelta(hours=2)
        pre_seg = raw_idx.loc[pre_start:s, 'active_power'].values.astype(np.float32)
        if len(pre_seg) > 0:
            sb = pre_seg[(pre_seg >= STANDBY_MIN_W) & (pre_seg < ON_THRESHOLD)]
            dur_min = len(sb) * sample_s / 60
            if len(sb) > 0 and dur_min >= STANDBY_MIN_MIN:
                standby = {
                    'duration_min': round(dur_min, 1),
                    'avg_w':        round(float(sb.mean()), 2),
                    'energy_wh':    round(float(sb.mean()) * dur_min / 60, 3),
                }

        event = make_short_term_event(app_name, seg, ts, standby)
        short_term_events.append(event)

    print(f'  → {len(periods)}개 ON 구간 처리 완료')

print(f'\n총 {len(short_term_events)}개 이벤트 생성')

In [ ]:
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(short_term_events, f, ensure_ascii=False, indent=2, default=str)

print(f'저장 완료: {OUTPUT_PATH}')
print(f'총 이벤트 수: {len(short_term_events)}')

# 요약
by_app = defaultdict(list)
for e in short_term_events:
    by_app[e['appliance']].append(e)
print('\n가전별 이벤트 수:')
for app, events in sorted(by_app.items()):
    modes = list({e['mode'] for e in events})
    print(f'  {app}: {len(events)}개  모드={modes}')